# CPU-1: Byte-Level LM - Training en 2xT4

Entrena el modelo CPU-1 (~170M parametros ternarios) con knowledge distillation desde un teacher BPE (Qwen-14B).

**Requisito:** Seleccionar **GPU T4 x2** en Kaggle Settings -> Accelerator.

In [ ]:
import os, sys, subprocess, pathlib, torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")

n_gpus = torch.cuda.device_count()
print(f"\nGPUs disponibles: {n_gpus}")
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  ({p.total_memory / 1024**3:.1f} GB)")

if n_gpus < 2:
    print("\nAVISO: Menos de 2 GPUs detectadas.")
    print("   Ir a Settings -> Accelerator -> GPU T4 x2")
else:
    print("\nOK: 2 GPUs listas para DDP")

In [ ]:
# HuggingFace token para subir checkpoints automaticamente.
# Crear un secreto llamado HF_TOKEN en Kaggle:
#   Add-ons -> Secrets -> Add New Secret  (nombre: HF_TOKEN, valor: hf_...)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("OK: HF_TOKEN configurado desde Kaggle Secrets")
except Exception as e:
    print(f"AVISO: HF_TOKEN no encontrado ({e})")
    print("   Los checkpoints solo se guardaran localmente en /kaggle/working/")

In [ ]:
# Directorio de salida — persiste entre sesiones en /kaggle/working/
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Checkpoints -> {CHECKPOINT_DIR}")

# Verificar que los archivos del modelo estan disponibles
for f in ["model_v2.py", "train.py"]:
    path = pathlib.Path(f"/kaggle/working/{f}")
    if path.exists():
        print(f"OK: {f}")
    else:
        print(f"ERROR: {f} NO encontrado en /kaggle/working/")

## Dry-run — verificar que todo anda antes de entrenar con datos reales

In [ ]:
cmd_dry = [
    "torchrun",
    f"--nproc_per_node={n_gpus}",
    "--master_port=12355",
    "/kaggle/working/train.py",
    "--dry-run",
    "--dry-run-samples", "200",
    "--total-steps", "30",
    "--log-every", "5",
    "--batch-per-gpu", "2",
    "--gradient-accumulation-steps", "1",
    "--num-workers", "0",
    "--output-dir", CHECKPOINT_DIR,
]

print("Comando:", " ".join(cmd_dry))
print("-" * 60)
result = subprocess.run(cmd_dry)
print("-" * 60)
if result.returncode == 0:
    print("OK: Dry-run completado — el setup funciona correctamente")
else:
    print(f"ERROR: Dry-run fallo (exit code {result.returncode})")

## Entrenamiento Real

**Antes de correr esta celda:**
1. Adjuntar el dataset como input en Kaggle (Dataset -> Add Data)
2. Ajustar `DATA_PATH` con la ruta correcta al dataset
3. Configurar `HF_TOKEN` en Kaggle Secrets si queres subir checkpoints a HuggingFace

El dataset debe tener columnas: `patches`, `targets`, `teacher_probs`, `teacher_mask`
(generado por `byte_marginalization.py`)

In [ ]:
# ============================================================
# ENTRENAMIENTO REAL — ajustar DATA_PATH y descomentar
# ============================================================

# DATA_PATH = "/kaggle/input/cpu1-dataset"   # <-- ajustar al nombre del dataset adjunto
# RESUME_FROM = None                          # o "checkpoint_5000.pt" para resumir

# cmd_train = [
#     "torchrun",
#     f"--nproc_per_node={n_gpus}",
#     "--master_port=12355",
#     "/kaggle/working/train.py",
#     "--data-path", DATA_PATH,
#     "--total-steps", "100000",
#     "--warmup-steps", "2000",
#     "--log-every", "100",
#     "--save-every", "500",
#     "--batch-per-gpu", "4",
#     "--gradient-accumulation-steps", "2",
#     "--gradient-checkpointing",
#     "--num-workers", "4",
#     "--upload-hf",
#     "--output-dir", CHECKPOINT_DIR,
# ]

# # Para resumir desde checkpoint:
# # if RESUME_FROM:
# #     cmd_train += ["--resume-from", f"{CHECKPOINT_DIR}/{RESUME_FROM}"]

# print("Comando:", " ".join(cmd_train))
# print("-" * 60)
# result = subprocess.run(cmd_train)
# print(f"Exit code: {result.returncode}")

print("Celda comentada — leer instrucciones arriba antes de ejecutar")